In [4]:
import numpy as np    
from sympy import symbols, nsolve, cos, sin, sqrt, pi
from collections import defaultdict
import os
from PIL import Image, ImageDraw, ImageFont
import imageio
import matplotlib.pyplot as plt

In [21]:
def calculate_energy_levels(N, y1, y2, a, E_min, E_max, k=10):
    """
    计算给定参数下的能量级别(E值)
    参数:
        N: 系统参数
        y1, y2: 势能参数
        a: 系统参数
        E_min, E_max: 能量范围
        k: 合并解的精度（小数点后k位相同则合并）
    返回:
        排序后的唯一E值列表
    """
    N_0 = N if N % 2 == 0 else N - 1
    n_values = np.arange(int(N_0))
    n_values = n_values[1:]
    E_sym = symbols('E')
    all_solutions = []

    for n in n_values:
        equation = cos(2*a*sqrt(E_sym)) + y1*sin(2*a*sqrt(E_sym))/sqrt(E_sym) \
                  - (y1**2 + y2**2)*(cos(2*a*sqrt(E_sym))-1)/(4*E_sym) \
                  - cos(1*pi*(n-0.5)/N)
        
        solutions = []
        for E_guess in np.linspace(E_min + 0.1, E_max, 100):
            try:
                sol = float(nsolve(equation, E_guess, tol=1e-12))
                if E_min <= sol <= E_max:
                    solutions.append(sol)
            except:
                continue
        
        # 合并解（保留小数点后k位相同的唯一值）
        merged = sorted(list({round(sol, k) for sol in solutions}))
        all_solutions.extend(merged)
    
    return sorted(all_solutions)

def calculate_energy_levels_New(N, y1, y2, a, E_min, E_max, k=10):

    N_0 = N if N % 2 == 0 else N - 1
    n_values = np.arange(int(N_0 + 1))
    n_values = n_values[1:]
    E_sym = symbols('E')
    all_solutions = []

    for n in n_values:
        equation = cos(2*a*sqrt(E_sym)) + y1*sin(2*a*sqrt(E_sym))/sqrt(E_sym) \
                  - (y1**2 + y2**2)*(cos(2*a*sqrt(E_sym))-1)/(4*E_sym) \
                  - cos(1*pi*(n)/N)
        
        solutions = []
        for E_guess in np.linspace(E_min + 0.1, E_max, 100):
            try:
                sol = float(nsolve(equation, E_guess, tol=1e-12))
                if E_min <= sol <= E_max:
                    solutions.append(sol)
            except:
                continue
        merged = sorted(list({round(sol, k) for sol in solutions}))
        all_solutions.extend(merged)
    
    return sorted(all_solutions)

def coupling(matrix_1, matrix_2):
    s11_1, s12_1 = matrix_1[0, 0], matrix_1[0, 1]
    s21_1, s22_1 = matrix_1[1, 0], matrix_1[1, 1]
    s11_2, s12_2 = matrix_2[0, 0], matrix_2[0, 1]
    s21_2, s22_2 = matrix_2[1, 0], matrix_2[1, 1]
    
    denom = 1 - s11_2 * s22_1
    s11 = s11_1 + s12_1 * s21_1 * s11_2 / denom
    s12 = s12_1 * s12_2 / denom
    s21 = s21_2 * s21_1 / denom
    s22 = s22_2 + s21_2 * s12_2 * s22_1 / denom
    
    return np.array([[s11, s12], [s21, s22]], dtype=np.complex128)

def multiple_coupling(matrix_collect):
    result = matrix_collect[0]
    for matrix in matrix_collect[1:]:
        result = coupling(result, matrix)
    return result

def coupling_matrix_of_N_modified(E, y1, y2, N, n3, n4, a):
    constant_1 = 2 * m * (y1 + 1j * y2) / (h1**2)
    constant_2 = 2 * m * (y1 - 1j * y2) / (h1**2)
    
    di = np.array([constant_1 if i % 2 == 0 or i >= 2 * N and i < 2 * N + n3 else constant_2 
                   for i in range(2 * N + n3 + n4)], dtype=complex)
    
    x_0 = 0
    x_1 = a * (2 * N + n3 + n4) - a + x_0
    location = np.linspace(x_0, x_1, 2 * N + n3 + n4)
    k = np.sqrt(E * 2 * m) / h1
    exp_2jk = np.exp(2j * k * location)
    exp_minus_2jk = np.exp(-2j * k * location)
    
    matrix_collect = np.zeros((2 * N + n3 + n4, 2, 2), dtype=complex)
    for i in range(2 * N + n3 + n4):
        s11 = di[i] * exp_2jk[i]
        s12 = 2j * k
        s21 = 2j * k
        s22 = di[i] * exp_minus_2jk[i]
        
        matrix_collect[i] = np.array([[s11, s12], [s21, s22]]) / (-di[i] + 2j * k)
    return multiple_coupling(matrix_collect)

def Transmission_coefficient_of_N_modified(E, y1, y2, N, n3, n4, a):
    matrix = coupling_matrix_of_N_modified(E, y1, y2, N, n3, n4, a)
    return float(np.abs(matrix[1, 0])**2)

def Reflection_coefficient_of_N_modified(E, y1, y2, N, n3, n4, a):
    matrix = coupling_matrix_of_N_modified(E, y1, y2, N, n3, n4, a)
    return float(np.abs(matrix[0, 0])**2)

def create_folder_name(y11, y22, N_set, E_ranges, k, zhi_1,E_point):
    y1_str = f"y1_{'_'.join(map(str, y11))}"
    y2_str = f"y2_{'_'.join(map(str, y22))}"
    N_str = f"N_{N_set[0]}-{N_set[-1]}"
    E_str = f"E_{E_ranges[0][0]}-{E_ranges[0][1]}"
    k_str = f"k_{k}"
    return f"T&R-25.4.25"                                         ##{y1_str}_{y2_str}_{N_str}_{E_str}_{k_str}_{zhi_1}_{E_point}

def create_gif_name(y11, y22, N_set, E_ranges, k, zhi_1,E_point):
    y1_str = f"y1_{'_'.join(map(str, y11))}"
    y2_str = f"y2_{'_'.join(map(str, y22))}"
    N_str = f"N_{N_set[0]}-{N_set[-1]}"
    E_str = f"E_{E_ranges[0][0]}-{E_ranges[0][1]}"
    k_str = f"k_{k}"
    return f"T&R_Animation_{y1_str}_{y2_str}_{N_str}_{E_str}_{k_str}_{zhi_1}_{E_point}.gif"



In [22]:
def generate_results(y11, y22, a, zhi_1, E_ranges, n3, n4, m, h1, N_set, k=10, zhi_2_shengchengdongtu=['否'],E_point=10000):
    """
    主函数，生成所有结果
    参数:
        y11, y22: 势能参数列表
        a: 系统参数
        zhi_1: 计算类型列表（['T']或['R']或['T+R']）
        E_ranges: 能量范围列表
        n3, n4: 系统参数
        m, h1: 物理常数
        N_set: N值集合
        k: 合并解的精度
        zhi_2_shengchengdongtu: 是否生成动态图（['是']或['否']）
    """
    # 主输出目录
    main_output_dir = r"D:\结果"
    os.makedirs(main_output_dir, exist_ok=True)
    
    # 创建基于参数的文件夹
    folder_name = create_folder_name(y11, y22, N_set, E_ranges, k ,zhi_1,E_point)
    output_dir = os.path.join(main_output_dir, folder_name)
    os.makedirs(output_dir, exist_ok=True)
    
    # 生成所有静态图像
    for N in N_set:
        # 计算能量级别
        energy_levels = calculate_energy_levels(N, y11[0], y22[0], a, E_ranges[0][0], E_ranges[0][1], k)
        energy_levels_New = calculate_energy_levels_New(N, y11[0], y22[0], a, E_ranges[0][0], E_ranges[0][1], k)

        # 计算传输/反射系数
        E_values = np.linspace(E_ranges[0][0], E_ranges[0][1], E_point)
        
        fig, ax = plt.subplots(figsize=(12, 8))  # 增大图形尺寸
        
        if 'T' in zhi_1:
            T_values = [Transmission_coefficient_of_N_modified(E, y11[0], y22[0], N, n3, n4, a) for E in E_values]
            ax.plot(E_values, T_values, 
                   label='Transmission (T)' ,
                   color='blue',          # 改色
                   linewidth=2,            # 加粗线条
                   alpha=0.9)
        
        if 'R' in zhi_1:
            R_values = [Reflection_coefficient_of_N_modified(E, y11[0], y22[0], N, n3, n4, a) for E in E_values]
            ax.plot(E_values, R_values, 
                   label='Reflection (R)', 
                   color='red',          # 改色
                   linewidth=1.5,            # 加粗线条
                   alpha=0.9,
                   linestyle='-')         # 使用虚线区分
            
        if 'T+R' in zhi_1:
            if 'T' in zhi_1 and 'R' in zhi_1:  # 确保两者都已计算
                Sum_values = [t + r for t, r in zip(T_values, R_values)]
                ax.plot(E_values, Sum_values, label='T + R', color='gray', linestyle='--', linewidth=0.9)
        
        # 添加能量级别线
        for x in energy_levels:
            ax.axvline(x, color='green', linestyle='-', linewidth=0.4, alpha=0.5)
        
        for x in energy_levels_New:
            ax.axvline(x, color='red', linestyle='-', linewidth=0.4, alpha=0.5)

        # 设置图形属性
        ax.set_xlabel('Energy (E)', fontsize=14, fontweight='bold')
        ax.set_ylabel('Coefficient', fontsize=14, fontweight='bold')
        ax.set_title(f'Transmission/Reflection Coefficient\n(N={N}, y1={y11[0]}, y2={y22[0]}, k={k})', 
                    fontsize=16, pad=20)
        
        # 自定义图例（右上角，大字体）
        ax.legend(loc='upper right', 
                 fontsize=12,
                 framealpha=1,
                 edgecolor='black',
                 facecolor='white',
                 bbox_to_anchor=(0.98, 0.98))  # 精确定位右上角
        
        # 坐标轴范围设置
        ax.set_xlim(E_ranges[0][0], E_ranges[0][1])
        ax.set_ylim(0, 2)

        # 网格线设置
#        ax.grid(False, 
#        linestyle='-',  # 实线
#        color='black',   # 蓝色
#        alpha=0.2,      # 更低的透明度
#        linewidth=0.8)  # 适当粗细
        
        # 调整边距
        plt.subplots_adjust(right=0.85, top=0.85)
        
        # 保存图片
        img_name = f"T&R_N_{N}_y1_{y11[0]}_y2_{y22[0]}_k_{k}_E_{E_ranges[0][0]}-{E_ranges[0][1]}-{zhi_1}.png"
        img_path = os.path.join(output_dir, img_name)
        plt.savefig(img_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"已保存: {img_path}")
    
    # 如果需要生成动态图
    if zhi_2_shengchengdongtu == ['是']:
        gif_name = create_gif_name(y11, y22, N_set, E_ranges, k,zhi_1,E_point)
        gif_path = os.path.join(main_output_dir, gif_name)
        
        # 获取所有图片文件并按N值排序
        image_files = [f for f in os.listdir(output_dir) if f.endswith(".png")]
        image_files.sort(key=lambda x: int(x.split("N_")[1].split("_")[0]))
        
        # 准备字体
        try:
            font = ImageFont.truetype("arial.ttf", 36)  # 增大字体
        except:
            font = ImageFont.load_default()
        
        # 处理每张图片并创建帧列表
        frames = []
        for img_file in image_files:
            img_path = os.path.join(output_dir, img_file)
            
            with Image.open(img_path) as img:
                if img.mode in ('RGBA', 'LA'):
                    background = Image.new('RGB', img.size, (255, 255, 255))
                    background.paste(img, mask=img.split()[-1])
                    img = background
                
                draw = ImageDraw.Draw(img)
                N_value = img_file.split("N_")[1].split("_")[0]
                text = f"N = {N_value}"
                
                # 计算文本位置（右上角）
                bbox = draw.textbbox((0, 0), text, font=font)
                text_width = bbox[2] - bbox[0]
                text_height = bbox[3] - bbox[1]
                margin = 15
                x = img.width - text_width - margin
                y = margin
                
                # 绘制背景框和文本（更醒目的样式）
                draw.rectangle([x-margin, y-margin, x+text_width+margin, y+text_height+margin], 
                              fill="black", outline="white", width=2)
                draw.text((x, y), text, fill="white", font=font)
                
                frames.append(img)
        
        # 保存为GIF（增加每帧显示时间）
        imageio.mimsave(gif_path, frames, format='GIF', duration=2)  # 延长到2.5秒/帧
        print(f"动态图已保存至: {gif_path}")

In [23]:
if __name__ == "__main__":
    # 系统参数
    y11 = [2]
    y22 = [2]
    a = 0.5
    zhi_1 = ['R','T','T+R']#,'R','T+R']
    E_ranges = [(165, 168)]
    n3 = 0
    n4 = 0
    m = 0.5
    h1 = 1
    N_set = [100]#np.linspace(100, 200, 200).astype(int)
    k = 10  # 合并精度
    zhi_2_shengchengdongtu = ['否']  # 是否生成动态图
    E_point=2000
    
    generate_results(y11, y22, a, zhi_1, E_ranges, n3, n4, m, h1, N_set, k, zhi_2_shengchengdongtu,E_point)

已保存: D:\结果\T&R-25.4.25\T&R_N_100_y1_2_y2_2_k_10_E_165-168-['R', 'T', 'T+R'].png


In [8]:
if __name__ == "__main__":
    # 系统参数
    y11 = [2]
    y22 = [2]
    a = 0.5
    zhi_1 = ['R','T','T+R']#,'R','T+R']
    E_ranges = [(168, 180)]
    n3 = 0
    n4 = 0
    m = 0.5
    h1 = 1
    N_set = [100]#np.linspace(100, 200, 200).astype(int)
    k = 10  # 合并精度
    zhi_2_shengchengdongtu = ['否']  # 是否生成动态图
    E_point=1000
    
    generate_results(y11, y22, a, zhi_1, E_ranges, n3, n4, m, h1, N_set, k, zhi_2_shengchengdongtu,E_point)

已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_25.4.3\T&R-25.4.25\T&R_N_100_y1_2_y2_2_k_10_E_168-180-['R', 'T', 'T+R'].png


In [18]:
if __name__ == "__main__":
    # 系统参数
    y11 = [2]
    y22 = [2]
    a = 0.5
    zhi_1 = ['R','T','T+R']#,'R','T+R']
    E_ranges = [(180, 230)]
    n3 = 0
    n4 = 0
    m = 0.5
    h1 = 1
    N_set = [100]#np.linspace(100, 200, 200).astype(int)
    k = 10  # 合并精度
    zhi_2_shengchengdongtu = ['否']  # 是否生成动态图
    E_point=1000
    
    generate_results(y11, y22, a, zhi_1, E_ranges, n3, n4, m, h1, N_set, k, zhi_2_shengchengdongtu,E_point)

已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_25.4.3\T&R-25.4.25\T&R_N_100_y1_2_y2_2_k_10_E_180-230-['R', 'T', 'T+R'].png


In [8]:
if __name__ == "__main__":
    # 系统参数
    y11 = [2]
    y22 = [2]
    a = 0.5
    zhi_1 = ['R','T','T+R']#,'R','T+R']
    E_ranges = [(230, 270)]
    n3 = 0
    n4 = 0
    m = 0.5
    h1 = 1
    N_set = [100]#np.linspace(100, 200, 200).astype(int)
    k = 10  # 合并精度
    zhi_2_shengchengdongtu = ['否']  # 是否生成动态图
    E_point=1000
    
    generate_results(y11, y22, a, zhi_1, E_ranges, n3, n4, m, h1, N_set, k, zhi_2_shengchengdongtu,E_point)

已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_25.4.3\T&R-25.4.25\T&R_N_100_y1_2_y2_2_k_10_E_230-270-['R', 'T', 'T+R'].png


In [3]:

if __name__ == "__main__":
    # 系统参数
    y1=2
    y2=2
    a=0.5
    E_min=230
    E_max=270
    N=100
    calculate_energy_levels(N, y1, y2, a, E_min, E_max, k=10)

In [ ]:

if __name__ == "__main__":
    # 系统参数
    y1=2
    y2=2
    a=0.5
    E_min=230
    E_max=270
    N=100
    calculate_energy_levels(N, y1, y2, a, E_min, E_max, k=10)
print()

In [21]:
if __name__ == "__main__":
    # 系统参数
    y11 = [2]
    y22 = [2]
    a = 0.5
    zhi_1 = ['R','T','T+R']#,'R','T+R']
    E_ranges = [(270, 352)]
    n3 = 0
    n4 = 0
    m = 0.5
    h1 = 1
    N_set = [100]#np.linspace(100, 200, 200).astype(int)
    k = 10  # 合并精度
    zhi_2_shengchengdongtu = ['否']  # 是否生成动态图
    E_point=1000
    
    generate_results(y11, y22, a, zhi_1, E_ranges, n3, n4, m, h1, N_set, k, zhi_2_shengchengdongtu,E_point)

已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_25.4.3\T&R-25.4.25\T&R_N_100_y1_2_y2_2_k_10_E_270-352-['R', 'T', 'T+R'].png


In [20]:
if __name__ == "__main__":
    # 系统参数
    y11 = [2]
    y22 = [2]
    a = 0.5
    zhi_1 = ['R','T','T+R']#,'R','T+R']
    E_ranges = [(352, 358)]
    n3 = 0
    n4 = 0
    m = 0.5
    h1 = 1
    N_set = [100]#np.linspace(100, 200, 200).astype(int)
    k = 10  # 合并精度
    zhi_2_shengchengdongtu = ['否']  # 是否生成动态图
    E_point=2000
    
    generate_results(y11, y22, a, zhi_1, E_ranges, n3, n4, m, h1, N_set, k, zhi_2_shengchengdongtu,E_point)

已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_25.4.3\T&R-25.4.25\T&R_N_100_y1_2_y2_2_k_10_E_352-358-['R', 'T', 'T+R'].png


In [24]:
def calculate_energy_levels1(N):
    """
    计算给定参数下的能量级别(E值)
    参数:
        N: 系统参数
        y1, y2: 势能参数
        a: 系统参数
        E_min, E_max: 能量范围
        k: 合并解的精度（小数点后k位相同则合并）
    返回:
        排序后的唯一E值列表
    """
    N_0 = N if N % 2 == 0 else N - 1
    n_values = np.arange(int(N_0))
    n_values= n_values[1:]
    return n_values
N=100
calculate_energy_levels1(N)

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51,
       52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68,
       69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85,
       86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99])